In [ ]:
#TAME/
#├── selected_participants_20.csv
#├── selected_audio_files_20_selection.csv
#├── selected_audio_files_20_original.csv
#├── selected_audio_files_20_wiener.csv
#├── selected_audio_files_20_pain_balanced_selection.csv
#├── selected_audio_files_20_pain_balanced_original.csv
#├── selected_audio_files_20_pain_balanced_wiener.csv
#├── selected_original_audio_20/
#├── selected_wiener_audio_20/
#├── selected_original_audio_20_pain_balanced/
#└── selected_wiener_audio_20_pain_balanced/

In [1]:
from pathlib import Path
import os
import shutil
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import wiener

# =========================
# Paths
# =========================
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")
AUDIO_PATH = DATA_PATH / "mic1_trim_v1~"
AUDIO_BASE_PATH = AUDIO_PATH / "mic1_trim_v2"
META_AUDIO_PATH = DATA_PATH / "meta_audio.csv"
META_PARTICIPANT_PATH = DATA_PATH / "meta_participant.csv"

assert DATA_PATH.exists(), "Data map niet gevonden!"
assert AUDIO_PATH.exists(), "Audio map niet gevonden!"
assert AUDIO_BASE_PATH.exists(), "Audio base path niet gevonden!"
assert META_AUDIO_PATH.exists(), "meta_audio.csv niet gevonden!"
assert META_PARTICIPANT_PATH.exists(), "meta_participant.csv niet gevonden!"

print("The paths are correct")


# =========================
# Helpers
# =========================
def load_csv_flexible(file_path):
    df = pd.read_csv(file_path)

    if df.shape[1] == 1:
        first_col = df.columns[0]
        split_df = df[first_col].str.split(",", expand=True)
        split_df.columns = [col.strip() for col in first_col.split(",")]
        df = split_df.copy()

    df.columns = [col.strip() for col in df.columns]
    return df


def stratified_participant_sample(
    file_path,
    n=20,
    gender_col="GENDER",
    race_col="RACE/ETHNICITY",
    id_col="PID",
    random_state=42
):
    df = load_csv_flexible(file_path).copy()

    needed_cols = [id_col, gender_col, race_col]
    for col in needed_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available columns: {list(df.columns)}")

    df = df.dropna(subset=[gender_col, race_col]).copy()

    if n > len(df):
        raise ValueError(f"You asked for {n} participants, but only {len(df)} are available.")

    df["stratum"] = (
        df[gender_col].astype(str).str.strip() + " | " +
        df[race_col].astype(str).str.strip()
    )

    stratum_counts = df["stratum"].value_counts().sort_index()
    proportions = stratum_counts / len(df)

    target_counts = proportions * n
    allocated = np.floor(target_counts).astype(int)

    remainder = n - allocated.sum()
    fractional_parts = (target_counts - allocated).sort_values(ascending=False)

    for stratum in fractional_parts.index[:remainder]:
        allocated[stratum] += 1

    allocated = pd.Series({
        stratum: min(allocated[stratum], stratum_counts[stratum])
        for stratum in allocated.index
    })

    current_total = allocated.sum()
    if current_total < n:
        shortfall = n - current_total
        spare_capacity = pd.Series({
            stratum: stratum_counts[stratum] - allocated[stratum]
            for stratum in allocated.index
        }).sort_values(ascending=False)

        for stratum in spare_capacity.index:
            if shortfall == 0:
                break
            if spare_capacity[stratum] > 0:
                extra = min(spare_capacity[stratum], shortfall)
                allocated[stratum] += extra
                shortfall -= extra

    sampled_parts = []
    for stratum, k in allocated.items():
        if k > 0:
            subset = df[df["stratum"] == stratum]
            sampled_parts.append(subset.sample(n=k, random_state=random_state))

    sample_df = (
        pd.concat(sampled_parts)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
        .drop(columns=["stratum"])
    )

    return sample_df


def print_sample_summary(sample_df, gender_col="GENDER", race_col="RACE/ETHNICITY", id_col="PID"):
    print(f"\nNumber of selected participants: {len(sample_df)}\n")
    print("Distribution gender:")
    print(sample_df[gender_col].value_counts(dropna=False))
    print("\nDistribution race/ethnicity:")
    print(sample_df[race_col].value_counts(dropna=False))
    print("\nSelected participants:")
    print(sample_df[id_col].tolist())
    print()


def select_valid_audio_rows(meta_audio_path, selected_participants_df):
    meta_audio_df = load_csv_flexible(meta_audio_path).copy()
    selected_pids = selected_participants_df["PID"].astype(str).tolist()

    filtered_df = meta_audio_df[meta_audio_df["PID"].astype(str).isin(selected_pids)].copy()
    filtered_df["NOTES"] = filtered_df["NOTES"].fillna("").astype(str).str.strip()
    filtered_df["ACTION LABEL"] = pd.to_numeric(filtered_df["ACTION LABEL"], errors="coerce")

    filtered_df = filtered_df[
        (filtered_df["NOTES"] == "") &
        (filtered_df["ACTION LABEL"] == 0)
    ].copy()

    return filtered_df


def balance_audio_by_pain_score_soft(
    df,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
):
    data = df.copy()
    data[pain_col] = pd.to_numeric(data[pain_col], errors="coerce")
    data = data.dropna(subset=[pain_col]).copy()
    data[pain_col] = data[pain_col].astype(int)

    counts = data[pain_col].value_counts().sort_index()
    print("\nOriginal pain distribution:")
    print(counts)

    n_classes = len(counts)
    ideal_per_class = target_total // n_classes
    max_per_class = int(ideal_per_class * max_ratio)

    sampled_parts = []
    for pain_level, group in data.groupby(pain_col):
        n_samples = min(len(group), max_per_class)
        sampled_parts.append(group.sample(n=n_samples, random_state=random_state))

    balanced_df = pd.concat(sampled_parts)

    if len(balanced_df) < target_total:
        remaining = data.drop(balanced_df.index)
        extra_needed = target_total - len(balanced_df)

        if len(remaining) > 0:
            extra_samples = remaining.sample(
                n=min(extra_needed, len(remaining)),
                random_state=random_state
            )
            balanced_df = pd.concat([balanced_df, extra_samples])

    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nFinal number of samples:", len(balanced_df))
    print("\nNew pain distribution:")
    print(balanced_df[pain_col].value_counts().sort_index())

    return balanced_df


def build_audio_paths(filtered_audio_df, audio_base_path, extension=".wav"):
    df = filtered_audio_df.copy()

    df["PID"] = df["PID"].astype(str).str.strip()
    df["COND"] = df["COND"].astype(str).str.strip()
    df["UTTNUM"] = df["UTTNUM"].astype(str).str.strip()
    df["UTTID"] = df["UTTID"].astype(str).str.strip()

    df["filename"] = (
        df["PID"] + "." +
        df["COND"] + "." +
        df["UTTNUM"] + "." +
        df["UTTID"] + extension
    )

    df["file_path"] = df.apply(
        lambda row: os.path.join(audio_base_path, row["PID"], row["filename"]),
        axis=1
    )
    df["file_exists"] = df["file_path"].apply(os.path.exists)

    return df


def make_output_path(input_path, output_root):
    participant_id = os.path.basename(os.path.dirname(input_path))
    filename = os.path.basename(input_path)

    participant_output_dir = os.path.join(output_root, participant_id)
    os.makedirs(participant_output_dir, exist_ok=True)

    return os.path.join(participant_output_dir, filename)


def copy_selected_audio(audio_selection_df, output_dir, output_col_name):
    os.makedirs(output_dir, exist_ok=True)
    copied_rows = []

    for _, row in audio_selection_df.iterrows():
        input_path = row["file_path"]

        try:
            output_path = make_output_path(input_path, output_dir)
            shutil.copy2(input_path, output_path)

            row_dict = row.to_dict()
            row_dict[output_col_name] = output_path
            copied_rows.append(row_dict)

        except Exception as e:
            print(f"Error copying {input_path}: {e}")

    return pd.DataFrame(copied_rows)


def load_wav_file(file_path):
    return wavfile.read(file_path)


def save_wav_file(file_path, sample_rate, signal):
    signal = np.clip(signal, -32768, 32767).astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def apply_wiener_filter(signal, mysize=29, noise=None):
    signal = signal.astype(np.float32)
    return wiener(signal, mysize=mysize, noise=noise)


def create_wiener_audio(audio_selection_df, output_dir, mysize=29):
    os.makedirs(output_dir, exist_ok=True)
    processed_rows = []

    for _, row in audio_selection_df.iterrows():
        input_path = row["file_path"]

        try:
            sample_rate, signal = load_wav_file(input_path)
            filtered_signal = apply_wiener_filter(signal, mysize=mysize)
            output_path = make_output_path(input_path, output_dir)
            save_wav_file(output_path, sample_rate, filtered_signal)

            row_dict = row.to_dict()
            row_dict["wiener_file_path"] = output_path
            processed_rows.append(row_dict)

        except Exception as e:
            print(f"Error processing {input_path}: {e}")

    return pd.DataFrame(processed_rows)


def run_dataset_pipeline(
    dataset_name,
    audio_rows_df,
    audio_base_path,
    data_path,
    save_csv_prefix,
    apply_pain_balance=False,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
):
    # balance on pain
    if apply_pain_balance:
        audio_rows_df = balance_audio_by_pain_score_soft(
            df=audio_rows_df,
            pain_col=pain_col,
            target_total=target_total,
            max_ratio=max_ratio,
            random_state=random_state
        )

    # build paths
    audio_selection = build_audio_paths(audio_rows_df, audio_base_path=audio_base_path)
    audio_selection_existing = audio_selection[audio_selection["file_exists"]].copy()

    print(f"\n[{dataset_name}] Number of selected audio rows: {len(audio_selection)}")
    print(f"[{dataset_name}] Number of existing files: {audio_selection_existing['file_exists'].sum()}")

    # save selection csv
    selection_csv = data_path / f"{save_csv_prefix}_selection.csv"
    audio_selection_existing.to_csv(selection_csv, index=False)

    # copy original selected audio
    original_output_dir = data_path / f"selected_original_audio_{dataset_name}"
    original_df = copy_selected_audio(
        audio_selection_df=audio_selection_existing,
        output_dir=original_output_dir,
        output_col_name="selected_original_file_path"
    )
    original_csv = data_path / f"{save_csv_prefix}_original.csv"
    original_df.to_csv(original_csv, index=False)

    # create Wiener version
    wiener_output_dir = data_path / f"selected_wiener_audio_{dataset_name}"
    wiener_df = create_wiener_audio(
        audio_selection_df=audio_selection_existing,
        output_dir=wiener_output_dir,
        mysize=29
    )
    wiener_csv = data_path / f"{save_csv_prefix}_wiener.csv"
    wiener_df.to_csv(wiener_csv, index=False)

    print(f"[{dataset_name}] Saved original folder: {original_output_dir}")
    print(f"[{dataset_name}] Saved Wiener folder: {wiener_output_dir}")
    print(f"[{dataset_name}] Saved CSVs:")
    print(f"  - {selection_csv.name}")
    print(f"  - {original_csv.name}")
    print(f"  - {wiener_csv.name}")

    return {
        "selection": audio_selection_existing,
        "original_df": original_df,
        "wiener_df": wiener_df,
        "original_output_dir": original_output_dir,
        "wiener_output_dir": wiener_output_dir,
        "selection_csv": selection_csv,
        "original_csv": original_csv,
        "wiener_csv": wiener_csv,
    }


# Run pipeline
# 1) Select 20 participants
sample_20 = stratified_participant_sample(
    file_path=META_PARTICIPANT_PATH,
    n=20,
    random_state=42
)

print_sample_summary(sample_20)
sample_20.to_csv(DATA_PATH / "selected_participants_20.csv", index=False)

# 2) Valid audio rows for those 20 participants
filtered_audio_20 = select_valid_audio_rows(
    meta_audio_path=META_AUDIO_PATH,
    selected_participants_df=sample_20
)

# 3) Dataset A: 20 participants, no pain balancing
results_20 = run_dataset_pipeline(
    dataset_name="20",
    audio_rows_df=filtered_audio_20,
    audio_base_path=AUDIO_BASE_PATH,
    data_path=DATA_PATH,
    save_csv_prefix="selected_audio_files_20",
    apply_pain_balance=False,
    random_state=42
)

# 4) Dataset B: 20 participants + pain balanced
results_20_pain = run_dataset_pipeline(
    dataset_name="20_pain_balanced",
    audio_rows_df=filtered_audio_20,
    audio_base_path=AUDIO_BASE_PATH,
    data_path=DATA_PATH,
    save_csv_prefix="selected_audio_files_20_pain_balanced",
    apply_pain_balance=True,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
)

print("\nFinished.")

The paths are correct

Number of selected participants: 20

Distribution gender:
GENDER
Woman         10
Man            8
Male           1
Non-binary     1
Name: count, dtype: int64

Distribution race/ethnicity:
RACE/ETHNICITY
Asian                11
White                 6
Hispanic/Latino       2
Two or more races     1
Name: count, dtype: int64

Selected participants:
['p80330', 'p28030', 'p60145', 'p64560', 'p79665', 'p68870', 'p68340', 'p77560', 'p59520', 'p79550', 'p91315', 'p15965', 'p68625', 'p18785', 'p97630', 'p10085', 'p20960', 'p62650', 'p37540', 'p94215']


[20] Number of selected audio rows: 1941
[20] Number of existing files: 1941


c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: divide by zero encountered in divide
  res *= (1 - noise / lVar)
c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: invalid value encountered in multiply
  res *= (1 - noise / lVar)


[20] Saved original folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_original_audio_20
[20] Saved Wiener folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_wiener_audio_20
[20] Saved CSVs:
  - selected_audio_files_20_selection.csv
  - selected_audio_files_20_original.csv
  - selected_audio_files_20_wiener.csv

Original pain distribution:
REVISED PAIN
1     913
2     107
3     148
4     273
5     122
6     144
7     138
8      71
9      20
10      5
Name: count, dtype: int64

Final number of samples: 1055

New pain distribution:
REVISED PAIN
1     150
2     107
3     148
4     150
5     122
6     144
7     138
8      71
9      20
10      5
Name: count, dtype: int64

[20_pain_balanced] Number of selected audio rows: 1055
[20_pain_balanced] Number of existing files: 1055


c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: divide by zero encountered in divide
  res *= (1 - noise / lVar)
c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: invalid value encountered in multiply
  res *= (1 - noise / lVar)


[20_pain_balanced] Saved original folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_original_audio_20_pain_balanced
[20_pain_balanced] Saved Wiener folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_wiener_audio_20_pain_balanced
[20_pain_balanced] Saved CSVs:
  - selected_audio_files_20_pain_balanced_selection.csv
  - selected_audio_files_20_pain_balanced_original.csv
  - selected_audio_files_20_pain_balanced_wiener.csv

Finished.
